In [1]:
import goatpy as gp
from spatialdata import read_zarr
import spatialdata_plot
import matplotlib.pyplot as plt
import scanpy as sc
import squidpy as sq
import pandas as pd
import napari
import numpy as np
from napari_spatialdata import Interactive

/Users/andrewcauser/anaconda3/envs/maldi_env/lib/python3.10/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/Users/andrewcauser/anaconda3/envs/maldi_env/lib/python3.10/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
/Users/andrewcauser/anaconda3/envs/maldi_env/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warning

In [2]:
"""
auto_align.py
=============
Auto-registration of a MALDI imzML dataset against a whole-slide H&E image.

Key design decisions
--------------------
1.  Registration is performed at MALDI native pixel size (default 10 um/px)
    so the H&E thumbnail is tiny and peak RAM stays manageable.

2.  Matching uses normalised grayscale cross-correlation (TM_CCOEFF_NORMED)
    on the TIC image vs inverted H&E intensity.

3.  A two-pass rotation search (coarse 0-360, then fine) finds the correct
    slide orientation without assuming any starting angle.

4.  Optional QuPath GeoJSON annotations can be passed via geojson_path.
    They are transformed using an analytically-derived affine matrix that
    exactly replicates PIL's rotate(angle, expand=True) behaviour, so
    annotation polygons land pixel-perfectly on the registered canvas.
"""

from __future__ import annotations

import os
import gc
import json
from functools import partial
from pathlib import Path
from typing import Optional, Union

import numpy as np
import pandas as pd
import anndata as ad
import geopandas as gpd
import cv2 as cv
from PIL import Image
from shapely.geometry import box, shape
from shapely import transform as shapely_transform

from spatialdata import SpatialData
from spatialdata.models import Image2DModel, PointsModel, ShapesModel, TableModel
from spatialdata.transformations import Identity

from pyimzml.ImzMLParser import ImzMLParser

import numpy as np
from pyimzml.ImzMLParser import ImzMLParser, getionimage
from joblib import Parallel, delayed
from functools import partial
import anndata as ad
import pandas as pd
import geopandas as gpd
from shapely.geometry import box
import numpy as np
from anndata import AnnData
from spatialdata import SpatialData
from spatialdata.models import PointsModel, Image2DModel, TableModel, ShapesModel
from spatialdata.transformations import Identity
import pkg_resources
import os



def parmap(f, X, nprocs=None):
    """
    Parallel map using joblib (more robust for Jupyter).
    
    Parameters
    ----------
    f : callable
        Function to apply to each element
    X : iterable
        Input data
    nprocs : int, optional
        Number of processes (default: -1, all CPUs)
    
    Returns
    -------
    list
        Results in same order as input
    """
    if nprocs is None:
        nprocs = -1  # Use all CPUs
    
    return Parallel(n_jobs=nprocs, backend='loky')(
        delayed(f)(x) for x in X
    )


def getimage(peak, path, tol = 0.1): 
    p =  ImzMLParser(path) #individual file pointers otherwise parsing is corrupted
    return getionimage(p, peak, tol=tol, reduce_func=max)
    

def rd_peaks(fn):
    data = []
    with open(fn) as f:
        f.readline() #header
        for line in f:
            ss = line.split()
            if ss[0].strip('"') == 'M': continue
            data.append(float(ss[1]))
    return data


def rd_peaks_from_package():

    # Try to get the file from the package
    peaks_path = pkg_resources.resource_filename('goatpy', 'data/PEAKS.csv')
    
    if not os.path.exists(peaks_path):
        raise FileNotFoundError(f"PEAKS.csv not found at {peaks_path}")
    
    with open(peaks_path, 'r') as f:
        data = []
        f.readline()  # skip header
        for line in f:
            ss = line.split()
            if ss[0].strip('"') == 'M': 
                continue
            data.append(float(ss[1]))
    return data


def glyco_spatialdata(imzml_path, peaks_path = None, tol = 0.1, pixel_size = 20):

    # Load Peaks
    if peaks_path is None:
        peaks = rd_peaks_from_package()
    else:
        peaks = rd_peaks(peaks_path)

    # Load ImzML data
    getimg = partial(getimage, path=imzml_path, tol = tol)

    spectra_all = np.stack(
        parmap(getimg, peaks, 10),
        axis=-1
    )

    # Load Spatial Info
    p = ImzMLParser(imzml_path)
    coords = np.array(p.coordinates)[:, :2]  # (x, y)
    coords = coords - 1  # convert from 1-based to 0-based indexing
    

    # Create AnnData Object
    spectra_flat = np.array([spectra_all[y-1, x-1, :] for x, y in coords])
    anndata = ad.AnnData(spectra_flat, dtype=np.float32)
    anndata.var_names = np.array(["%.1f" % p for p in peaks])
    anndata.obs_names = np.array(list(map(str, range(len(coords)))))
    anndata.obs["full_x"] = coords[:, 0]
    anndata.obs["full_y"] = coords[:, 1]

    anndata.obs["x"] = anndata.obs["full_x"] - anndata.obs["full_x"].min() 
    anndata.obs["y"] = anndata.obs["full_y"] - anndata.obs["full_y"].min()
    

    anndata.obsm["spatial"] = np.column_stack([anndata.obs["x"], anndata.obs["y"]])


    # Calculate Total Ion Count (TIC)
    anndata.obs["MPI"] = np.ravel(anndata.X.sum(axis=1))

    
    # Create SpatialData Object
    coords = pd.DataFrame({
        "x": [c for c in anndata.obs["x"]],
        "y": [c for c in anndata.obs["y"]],
    })

    coords["instance_id"] = coords.index.astype(str)      # unique ID for each pixel
    coords["region"] = "pixels"                  # must exist for TableModel

    df = pd.concat(
        [
            coords.reset_index(drop=True),
            pd.DataFrame(anndata.X, columns=("mz-" + anndata.var.index))
        ],
        axis=1
    )

    
    points = PointsModel.parse(df)

    gdf = centroids_to_pixel_squares(df, x_col="x", y_col="y", pixel_size=1.0)  
    
    shapes = ShapesModel.parse(
                gdf[["instance_id", "geometry"]],
                transformations={"global": Identity()},
                )


    sdata = SpatialData(points={"centroids": points},
                        shapes={"pixels": shapes})


    adata = AnnData(
        X=anndata.X,
        obs=coords,  # contains x, y, point_id, region
        var=pd.DataFrame(index=("mz-" + anndata.var.index))
    )

    adata.obs = pd.concat(
        [
            adata.obs.reset_index(drop=True),
            anndata.obs.drop(columns=["x", "y"]).iloc[:adata.n_obs].reset_index(drop=True)
        ],
        axis=1
    )

    coords = np.array(adata.obs[["x", "y"]])

    adata.obsm["spatial"] = coords
    adata.obs["instance_id"] = sdata["pixels"].index
    adata.obs["region"] = "pixels"
    adata.obs["region"].astype("category")



    table = TableModel.parse(
        adata,
        region="pixels",      # name of your PointsModel
        region_key="region",        # must exist in adata.obs
        instance_key="instance_id"     # unique per row
    )

    # --- 7. Add to SpatialData ---
    sdata.tables["maldi_adata"] = table

    return sdata


def centroids_to_pixel_squares(df, x_col="x", y_col="y", pixel_size=1.0):
    
    half = pixel_size / 2
    
    geometries = [
        box(x - half, y - half, x + half, y + half)
        for x, y in zip(df[x_col], df[y_col])
    ]
    
    gdf = gpd.GeoDataFrame(
        df.copy(),
        geometry=geometries,
    )
    
    return gdf


# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------

def _log(msg: str) -> None:
    try:
        import psutil
        rss = psutil.Process(os.getpid()).memory_info().rss / 1e9
        print(f"[{rss:.2f}GB] {msg}")
    except ImportError:
        print(msg)


# ---------------------------------------------------------------------------
# H&E loading
# ---------------------------------------------------------------------------

def _read_native_mpp(he_path: str) -> Optional[float]:
    """Read native microns-per-pixel from file metadata without loading pixels."""
    try:
        import openslide
        slide = openslide.OpenSlide(he_path)
        mpp_x = slide.properties.get(openslide.PROPERTY_NAME_MPP_X)
        mpp_y = slide.properties.get(openslide.PROPERTY_NAME_MPP_Y)
        slide.close()
        if mpp_x and mpp_y:
            return (float(mpp_x) + float(mpp_y)) / 2.0
    except Exception:
        pass
    return None


def _load_he_at_resolution(he_path: str,
                            target_mpp: float,
                            native_mpp: float) -> tuple[Image.Image, float]:
    """
    Load H&E at (or just finer than) target_mpp using openslide pyramid
    selection, then resize to exactly target_mpp.
    """
    ext = os.path.splitext(he_path)[1].lower()
    wsi_exts = {'.svs', '.ndpi', '.scn', '.czi', '.mrxs'}

    try:
        import openslide
        slide = openslide.OpenSlide(he_path)

        best_level = 0
        best_mpp   = native_mpp
        for lvl in range(slide.level_count):
            lvl_mpp = native_mpp * slide.level_downsamples[lvl]
            if lvl_mpp <= target_mpp * 1.05:
                best_level = lvl
                best_mpp   = lvl_mpp

        dims   = slide.level_dimensions[best_level]
        region = slide.read_region((0, 0), best_level, dims)
        img    = region.convert('RGB')
        slide.close()
        _log(f"  openslide level {best_level}: {dims[0]}x{dims[1]}  "
             f"{best_mpp:.3f} um/px  ({img.width*img.height*3/1e6:.0f} MB)")

        if abs(best_mpp - target_mpp) / target_mpp > 0.02:
            scale = best_mpp / target_mpp
            nw    = max(1, round(img.width  * scale))
            nh    = max(1, round(img.height * scale))
            img   = img.resize((nw, nh), Image.Resampling.LANCZOS)
            best_mpp = target_mpp
            _log(f"  Resized to {nw}x{nh}  {target_mpp:.3f} um/px")

        return img, best_mpp

    except Exception as e:
        if ext in wsi_exts:
            raise RuntimeError(
                f"\nFailed to open '{he_path}' with openslide: {e}\n\n"
                "SVS/NDPI files require openslide:\n"
                "  conda install -c conda-forge openslide openslide-python\n"
            ) from e

    img = Image.open(he_path).convert('RGB')
    _log(f"  PIL: {img.width}x{img.height}")
    scale = native_mpp / target_mpp
    nw    = max(1, round(img.width  * scale))
    nh    = max(1, round(img.height * scale))
    img   = img.resize((nw, nh), Image.Resampling.LANCZOS)
    _log(f"  Resized to {nw}x{nh}  {target_mpp:.3f} um/px  ({nw*nh*3/1e6:.0f} MB)")
    return img, target_mpp


# ---------------------------------------------------------------------------
# MALDI loading
# ---------------------------------------------------------------------------

def _load_spectra(imzml_path: str,
                  peaks: list[float],
                  chunk_size: int = 10,
                  crop_r: int = 0,
                  crop_c: int = 0) -> np.ndarray:
    from pyimzml.ImzMLParser import getionimage
    p0    = ImzMLParser(imzml_path)
    probe = getionimage(p0, peaks[0], tol=0.1, reduce_func=max)
    h     = probe.shape[0] - crop_r
    w     = probe.shape[1] - crop_c
    del probe

    out = np.zeros((h, w, len(peaks)), dtype=np.float32)
    for start in range(0, len(peaks), chunk_size):
        batch = peaks[start: start + chunk_size]
        imgs  = parmap(partial(getimage, path=imzml_path), batch,
                       nprocs=min(len(batch), 4))
        for j, img in enumerate(imgs):
            out[:, :, start + j] = img[crop_r:, crop_c:]
        del imgs
        _log(f"  Peaks {start+1}-{min(start+len(batch), len(peaks))} / {len(peaks)}")
    return out


def _read_maldi_pixel_size(imzml_path: str) -> Optional[float]:
    try:
        p = ImzMLParser(imzml_path)
        for key in ['pixel size (x)', 'pixel size x', 'pixel size']:
            val = p.imzmldict.get(key)
            if val is not None:
                return float(val)
    except Exception:
        pass
    return None


def _crop_offsets(spectra_sum: np.ndarray, cutoff: float = 0.5) -> tuple[int, int]:
    try:
        crop_c = int(max(np.where(np.sum(spectra_sum, axis=0) < cutoff)[0]))
        crop_r = int(max(np.where(np.sum(spectra_sum, axis=1) < cutoff)[0]))
        return crop_r, crop_c
    except (ValueError, IndexError):
        return 0, 0


# ---------------------------------------------------------------------------
# Image preparation
# ---------------------------------------------------------------------------

def _maldi_to_grayscale(tic: np.ndarray) -> np.ndarray:
    blurred = cv.GaussianBlur(tic, (3, 3), 0)
    mn, mx  = blurred.min(), blurred.max()
    norm    = (blurred - mn) / (mx - mn) if mx > mn else blurred * 0.0
    _log(f"  MALDI grayscale: {norm.shape}  mean={norm.mean():.3f}")
    return norm.astype(np.float32)


def _he_to_grayscale(he_img: Image.Image) -> np.ndarray:
    gray = np.array(he_img.convert('L'), dtype=np.float32)
    inv  = 255.0 - gray
    mn, mx = inv.min(), inv.max()
    norm   = (inv - mn) / (mx - mn) if mx > mn else inv * 0.0
    _log(f"  H&E grayscale: {norm.shape}  mean={norm.mean():.3f}")
    return norm.astype(np.float32)


# ---------------------------------------------------------------------------
# Registration
# ---------------------------------------------------------------------------

def _match_at_rotation(he_gray: np.ndarray,
                        maldi_gray: np.ndarray,
                        rotation: float,
                        canvas_shape: tuple[int, int]) -> tuple[float, tuple[int, int]]:
    he_pil  = Image.fromarray((he_gray * 255).astype(np.uint8))
    rot_pil = he_pil.rotate(rotation, expand=True, resample=Image.Resampling.BILINEAR)
    rot_arr = np.array(rot_pil, dtype=np.float32) / 255.0

    rh, rw = rot_arr.shape
    canvas = np.zeros(canvas_shape, dtype=np.float32)
    pr     = max(0, (canvas_shape[0] - rh) // 2)
    pc     = max(0, (canvas_shape[1] - rw) // 2)
    use_h  = min(rh, canvas_shape[0] - pr)
    use_w  = min(rw, canvas_shape[1] - pc)
    canvas[pr: pr + use_h, pc: pc + use_w] = rot_arr[:use_h, :use_w]

    if canvas.shape[0] < maldi_gray.shape[0] or canvas.shape[1] < maldi_gray.shape[1]:
        raise ValueError(
            f"H&E canvas ({canvas.shape[1]}x{canvas.shape[0]} px) is smaller than "
            f"MALDI template ({maldi_gray.shape[1]}x{maldi_gray.shape[0]} px).\n"
            f"Try passing maldi_pixel_um=10 or maldi_pixel_um=20 explicitly."
        )

    result           = cv.matchTemplate(canvas, maldi_gray, cv.TM_CCOEFF_NORMED)
    _, score, _, loc = cv.minMaxLoc(result)
    return float(score), (int(loc[1]), int(loc[0]))


def _register(he_gray: np.ndarray,
              maldi_gray: np.ndarray,
              coarse_step: int = 15,
              fine_range: float = 5.0,
              fine_step: float = 1.0,
              buffer_px: int = 150) -> tuple[float, tuple[int, int]]:
    canvas_h     = he_gray.shape[0] + buffer_px
    canvas_w     = he_gray.shape[1] + buffer_px
    canvas_shape = (canvas_h, canvas_w)

    coarse_rots = list(range(0, 360, coarse_step))
    _log(f"  Coarse: {len(coarse_rots)} rotations (0-360 step {coarse_step}) ...")

    best_score = -np.inf
    best_rot   = 0.0
    best_idx   = (0, 0)

    for rot in coarse_rots:
        score, idx = _match_at_rotation(he_gray, maldi_gray, rot, canvas_shape)
        _log(f"    {rot:5.1f}  score={score:.4f}")
        if score > best_score:
            best_score, best_rot, best_idx = score, float(rot), idx

    _log(f"  Best coarse: {best_rot}  score={best_score:.4f}")

    fine_rots = sorted({
        round(best_rot + d, 1)
        for d in np.arange(-fine_range, fine_range + fine_step, fine_step)
        if abs(d) > 1e-6
    })
    _log(f"  Fine: {len(fine_rots)} rotations (+-{fine_range} step {fine_step}) ...")

    for rot in fine_rots:
        score, idx = _match_at_rotation(he_gray, maldi_gray, rot, canvas_shape)
        _log(f"    {rot:5.1f}  score={score:.4f}")
        if score > best_score:
            best_score, best_rot, best_idx = score, rot, idx

    _log(f"  Final: {best_rot}  score={best_score:.4f}  offset={best_idx}")
    return best_rot, best_idx


# ---------------------------------------------------------------------------
# Annotation transform — analytical PIL-replica affine
# ---------------------------------------------------------------------------

def _pil_rotation_affine(src_w: float, src_h: float, rotation_deg: float) -> np.ndarray:
    """
    Return the 3x3 affine matrix that maps (x, y) in the pre-rotation image
    to (x, y) in the post-rotation expanded canvas — exactly replicating
    PIL's Image.rotate(rotation_deg, expand=True).

    PIL rotates counter-clockwise around the geometric centre (src_w/2, src_h/2).
    """
    theta = np.deg2rad(rotation_deg)
    cos_t = np.cos(theta)
    sin_t = np.sin(theta)

    # FIXED: PIL uses width/2, height/2 (continuous image rectangle [0,w]×[0,h])
    cx = src_w / 2.0
    cy = src_h / 2.0

    corners = np.array([[0, 0], [src_w, 0], [src_w, src_h], [0, src_h]], dtype=np.float64)
    dx = corners[:, 0] - cx
    dy = corners[:, 1] - cy

    rx = cos_t * dx - sin_t * dy + cx
    ry = sin_t * dx + cos_t * dy + cy

    tx = -rx.min()
    ty = -ry.min()

    off_x = -cos_t * cx + sin_t * cy + cx + tx
    off_y = -sin_t * cx - cos_t * cy + cy + ty

    return np.array([
        [cos_t, -sin_t, off_x],
        [sin_t,  cos_t, off_y],
        [0,      0,     1    ],
    ])

def _build_full_transform_matrix(
    he_pixel_um: float,
    reg_mpp: float,
    he_reg_w: int,
    he_reg_h: int,
    rotation_deg: float,
    canvas_pr: int,
    canvas_pc: int,
    img_upscaling: int,
) -> np.ndarray:
    """
    Single 3x3 affine: native H&E pixel (QuPath coords) -> final upscaled canvas.

    Pipeline (applied right-to-left in the matrix multiply):
      1. Scale     native H&E px -> reg_mpp  (factor = he_pixel_um / reg_mpp)
      2. Rotate    PIL CCW expand=True at reg resolution
                   Source dims = he_reg_w x he_reg_h  (already at reg_mpp)
      3. Place     shift into zero-padded canvas  (+canvas_pc col, +canvas_pr row)
      4. Upscale   x img_upscaling

    CRITICAL: step 2 passes he_reg_w / he_reg_h directly to _pil_rotation_affine.
    These are already at reg_mpp resolution. Do NOT multiply them by scale_to_reg
    again — that would shrink the rotation centre to near (0, 0) and produce a
    massive positional offset in the output coordinates.
    """
    scale_to_reg = he_pixel_um / reg_mpp

    # 1 — native H&E px -> reg px
    M_scale = np.array([
        [scale_to_reg, 0,            0],
        [0,            scale_to_reg, 0],
        [0,            0,            1],
    ], dtype=np.float64)

    # 2 — PIL rotation at reg resolution.
    #     he_reg_w / he_reg_h are already at reg_mpp — pass them directly.
    M_rot = _pil_rotation_affine(he_reg_w, he_reg_h, rotation_deg)

    # 3 — placement in zero-padded canvas
    M_place = np.array([
        [1, 0, canvas_pc],
        [0, 1, canvas_pr],
        [0, 0, 1        ],
    ], dtype=np.float64)

    # 4 — upscale to match stored H&E image
    M_up = np.array([
        [img_upscaling, 0,             0],
        [0,             img_upscaling, 0],
        [0,             0,             1],
    ], dtype=np.float64)

    return M_up @ M_place @ M_rot @ M_scale


def _transform_geojson(
    geojson_path: Union[str, Path],
    he_pixel_um: float,
    reg_mpp: float,
    he_reg_w: int,
    he_reg_h: int,
    rotation_deg: float,
    canvas_pr: int,
    canvas_pc: int,
    img_upscaling: int,
    classification_key: str = "classification",
) -> tuple[gpd.GeoDataFrame, np.ndarray]:
    """
    Transform QuPath GeoJSON annotations into the aligned upscaled canvas.

    Returns
    -------
    gdf : GeoDataFrame with transformed geometries
    M   : 3x3 float64 affine (native H&E px -> upscaled canvas)
          stored in sdata so add_qupath_annotations() can reuse it.
    """
    with open(geojson_path, "r") as f:
        geojson = json.load(f)

    features = geojson if isinstance(geojson, list) else geojson.get("features", [])
    if not features:
        raise ValueError(f"No features found in {geojson_path}")

    M = _build_full_transform_matrix(
        he_pixel_um  = he_pixel_um,
        reg_mpp      = reg_mpp,
        he_reg_w     = he_reg_w,
        he_reg_h     = he_reg_h,
        rotation_deg = rotation_deg,
        canvas_pr    = canvas_pr,
        canvas_pc    = canvas_pc,
        img_upscaling= img_upscaling,
    )

    a, b, tx = M[0, 0], M[0, 1], M[0, 2]
    d, e, ty = M[1, 0], M[1, 1], M[1, 2]

    def _apply(coords: np.ndarray) -> np.ndarray:
        x = coords[:, 0]
        y = coords[:, 1]
        return np.column_stack([a * x + b * y + tx,
                                d * x + e * y + ty])

    geoms, labels, names = [], [], []
    for feat in features:
        geom_raw = feat.get("geometry")
        if geom_raw is None:
            continue
        geom    = shape(geom_raw)
        geom_tf = shapely_transform(geom, _apply)
        geoms.append(geom_tf)
        props = feat.get("properties") or {}
        clf   = props.get("classification") or {}
        label = clf.get("name", "unknown") if isinstance(clf, dict) else str(clf)
        labels.append(label)
        names.append(props.get("name", ""))

    gdf = gpd.GeoDataFrame({classification_key: labels, "name": names}, geometry=geoms)
    return gdf, M


# ---------------------------------------------------------------------------
# Output: build full H&E canvas
# ---------------------------------------------------------------------------

def _build_full_output(he_img: Image.Image,
                        rotation: float,
                        buffer_px: int) -> tuple[np.ndarray, int, int]:
    """
    Rotate the H&E thumbnail and place it centred in a zero-padded canvas.

    Returns
    -------
    canvas : np.ndarray (H, W, 3) uint8
    pr     : int  -- row offset where rotated H&E top-left was placed
    pc     : int  -- col offset where rotated H&E top-left was placed
    """
    canvas_h = he_img.height + buffer_px
    canvas_w = he_img.width  + buffer_px

    he_rot  = he_img.rotate(rotation, expand=True, resample=Image.Resampling.BILINEAR)
    rh, rw  = he_rot.height, he_rot.width
    pr      = max(0, (canvas_h - rh) // 2)
    pc      = max(0, (canvas_w - rw) // 2)
    use_h   = min(rh, canvas_h - pr)
    use_w   = min(rw, canvas_w - pc)

    canvas  = np.zeros((canvas_h, canvas_w, 3), dtype=np.uint8)
    rot_arr = np.array(he_rot, dtype=np.uint8)
    canvas[pr: pr + use_h, pc: pc + use_w] = rot_arr[:use_h, :use_w]
    del he_rot, rot_arr

    _log(f"  H&E canvas: {canvas_w}x{canvas_h}  ({canvas.nbytes/1e6:.0f} MB)")
    _log(f"  Rotated H&E placed at pr={pr}, pc={pc}  (rotated size: {rw}x{rh})")
    return canvas, pr, pc


# ---------------------------------------------------------------------------
# SpatialData construction
# ---------------------------------------------------------------------------

def _build_spatialdata(spectra_all: np.ndarray,
                       peaks: list[float],
                       maldi_pixel_um: float,
                       he_canvas: np.ndarray,
                       maldi_offset_in_canvas: tuple[int, int],
                       reg_mpp: float,
                       crop_r: int,
                       crop_c: int,
                       img_upscaling: int = 10,
                       library_id: str = "spatial") -> SpatialData:

    maldi_h, maldi_w, n_peaks = spectra_all.shape
    scale         = maldi_pixel_um / reg_mpp
    local_off_r, local_off_c = maldi_offset_in_canvas

    us      = img_upscaling
    he_up_h = he_canvas.shape[0] * us
    he_up_w = he_canvas.shape[1] * us
    he_up   = np.array(
        Image.fromarray(he_canvas).resize(
            (he_up_w, he_up_h), Image.Resampling.NEAREST
        ),
        dtype=np.uint8,
    )
    _log(f"  H&E upscaled {us}x: {he_up_w}x{he_up_h}  ({he_up.nbytes/1e6:.0f} MB)")

    grid_r, grid_c = np.mgrid[0: maldi_h, 0: maldi_w]
    he_r = ((local_off_r + (grid_r.flatten() + 0.5) * scale) * us)
    he_c = ((local_off_c + (grid_c.flatten() + 0.5) * scale) * us)

    adata = ad.AnnData(spectra_all.reshape(-1, n_peaks).copy(), dtype=np.float32)
    adata.var_names = np.array(["%.1f" % pk for pk in peaks])
    adata.obs_names = np.array([str(i) for i in range(maldi_h * maldi_w)])

    yy, xx = np.mgrid[crop_r: maldi_h + crop_r, crop_c: maldi_w + crop_c]
    adata.obs["x"]   = xx.flatten()
    adata.obs["y"]   = yy.flatten()
    adata.obs["MPI"] = np.ravel(adata.X.sum(axis=1))

    adata.obsm["spatial"] = np.column_stack([he_c, he_r])
    adata.obs["he_x"]     = he_c
    adata.obs["he_y"]     = he_r

    adata.uns["spatial"] = {
        library_id: {
            "images": {"hires": he_up},
            "use_quality": "hires",
            "scalefactors": {
                "tissue_hires_scalef": 1.0,
                "spot_diameter_fullres": float(us),
            },
        }
    }

    pixel_idx = np.arange(maldi_h * maldi_w).astype(str)
    half      = us / 2.0
    geoms     = [
        box(float(c) - half, float(r) - half,
            float(c) + half, float(r) + half)
        for r, c in zip(he_r, he_c)
    ]
    gdf    = gpd.GeoDataFrame({"cell_id": pixel_idx}, geometry=geoms)
    shapes = ShapesModel.parse(gdf, transformations={"global": Identity()})

    pts_df    = pd.DataFrame({"x": he_c, "y": he_r, "cell_id": pixel_idx})
    centroids = PointsModel.parse(pts_df)

    image_cyx = np.transpose(he_up, (2, 0, 1))
    img_model = Image2DModel.parse(
        image_cyx, dims=("c", "y", "x"),
        transformations={"global": Identity()},
    )

    sdata = SpatialData(
        images={"he_image": img_model},
        points={"centroids": centroids},
        shapes={"pixels": shapes},
    )

    adata.obs["instance_id"] = sdata["pixels"].index
    adata.obs["region"]      = "pixels"
    adata.obs["region"]      = adata.obs["region"].astype("category")

    table = TableModel.parse(
        adata, region="pixels",
        region_key="region", instance_key="instance_id",
    )
    sdata["maldi_adata"] = table
    return sdata


# ---------------------------------------------------------------------------
# Public API
# ---------------------------------------------------------------------------

def load_and_align(
    imzml_path: str,
    he_path: str,
    peaks_path: Optional[str] = None,
    geojson_path: Optional[Union[str, Path]] = None,
    geojson_shapes_key: str = "annotations",
    geojson_classification_key: str = "classification",
    maldi_pixel_um: Optional[float] = None,
    he_pixel_um: Optional[float] = None,
    spectra_chunk_size: int = 10,
    coarse_rotation_step: int = 15,
    fine_rotation_range: float = 5.0,
    fine_rotation_step: float = 1.0,
    buffer_px: int = 150,
    img_upscaling: int = 10,
) -> SpatialData:
    """
    Load a MALDI imzML dataset and an H&E image, auto-register them,
    and return a merged SpatialData object.

    Parameters
    ----------
    imzml_path : str
        Path to the .imzML file.
    he_path : str
        Path to the H&E image.  SVS/NDPI require openslide.
    peaks_path : str or None
        Path to peaks CSV.  Uses bundled PEAKS.csv when None.
    geojson_path : str, Path, or None
        Optional path to a QuPath GeoJSON annotation export.  Coordinates
        must be in native H&E pixel space (QuPath default).
    geojson_shapes_key : str, default "annotations"
        Key under which annotations are stored in sdata.shapes.
    geojson_classification_key : str, default "classification"
        Column name for the QuPath class label in the resulting GeoDataFrame.
    maldi_pixel_um : float or None
        Native MALDI pixel size in um.  Auto-read from imzML when None.
    he_pixel_um : float or None
        Native H&E pixel size in um.  Auto-read from metadata when None.
    spectra_chunk_size : int
        Ion images loaded in parallel at once.
    coarse_rotation_step : int
        Degrees between candidates in the 0-360 coarse sweep.
    fine_rotation_range : float
        +/- degrees searched around the best coarse angle.
    fine_rotation_step : float
        Degree increment for fine search.
    buffer_px : int
        Extra canvas padding (px at reg resolution) for the rotation search.
    img_upscaling : int
        Each MALDI pixel is upscaled to img_upscaling x img_upscaling canvas
        pixels in the output.

    Returns
    -------
    SpatialData with:
        images['he_image']             -- full rotated H&E canvas
        shapes['pixels']               -- one square per MALDI pixel
        shapes[geojson_shapes_key]     -- annotations (if geojson_path given)
        points['centroids']            -- centroid of each MALDI pixel
        tables['maldi_adata']          -- AnnData with ion intensities
    """

    # ------------------------------------------------------------------
    # 0. Peaks
    # ------------------------------------------------------------------
    _log("Loading peaks ...")
    peaks = rd_peaks(peaks_path) if peaks_path else rd_peaks_from_package()
    _log(f"  {len(peaks)} peaks")

    # ------------------------------------------------------------------
    # 0b. H&E native pixel size
    # ------------------------------------------------------------------
    if he_pixel_um is None:
        he_pixel_um = _read_native_mpp(he_path)
        if he_pixel_um is None:
            try:
                _img = Image.open(he_path)
                tag_info = getattr(_img, 'tag_v2', {})
                xres = tag_info.get(282)
                unit = tag_info.get(296, 2)
                if xres is not None:
                    xres = xres[0] / xres[1] if isinstance(xres, tuple) else float(xres)
                    he_pixel_um = (10000.0 / xres) if unit == 3 else (25400.0 / xres)
                    _log(f"  H&E pixel size from TIFF tags: {he_pixel_um:.4f} um/px")
                _img.close()
            except Exception:
                pass
        if he_pixel_um is None:
            he_pixel_um = 0.2527
            _log(f"  WARNING: H&E pixel size unknown, assuming {he_pixel_um} um/px.")
        else:
            _log(f"  H&E native pixel size: {he_pixel_um:.4f} um/px")

    try:
        _he_probe = Image.open(he_path)
        _he_native_w, _he_native_h = _he_probe.size
        _he_probe.close()
    except Exception:
        _he_native_w, _he_native_h = 10000, 10000
    he_phys_w_um = _he_native_w * he_pixel_um
    he_phys_h_um = _he_native_h * he_pixel_um

    # ------------------------------------------------------------------
    # 0c. MALDI pixel size
    # ------------------------------------------------------------------
    if maldi_pixel_um is None:
        detected = _read_maldi_pixel_size(imzml_path)
        if detected is not None:
            _log(f"  MALDI pixel size from imzML metadata: {detected} um/px")
            _p_probe = ImzMLParser(imzml_path)
            _maldi_h = _p_probe.imzmldict.get('max count of pixels y', 1)
            _maldi_w = _p_probe.imzmldict.get('max count of pixels x', 1)
            _he_thumb_w = he_phys_w_um / detected
            _he_thumb_h = he_phys_h_um / detected
            if _he_thumb_w >= _maldi_w and _he_thumb_h >= _maldi_h:
                maldi_pixel_um = detected
                _log(f"  Validated: H&E thumbnail ({_he_thumb_w:.0f}x{_he_thumb_h:.0f} px) "
                     f">= MALDI ({_maldi_w}x{_maldi_h} px)")
            else:
                _log(f"  WARNING: imzML pixel size {detected} um makes H&E thumbnail "
                     f"({_he_thumb_w:.0f}x{_he_thumb_h:.0f} px) smaller than MALDI "
                     f"({_maldi_w}x{_maldi_h} px) -- likely wrong.")
                for candidate in [10.0, 20.0, 50.0, 100.0, 200.0]:
                    _cw = he_phys_w_um / candidate
                    _ch = he_phys_h_um / candidate
                    if _cw >= _maldi_w * 0.5 and _ch >= _maldi_h * 0.5:
                        maldi_pixel_um = candidate
                        _log(f"  Auto-selected maldi_pixel_um={candidate} um/px")
                        break
                if maldi_pixel_um is None:
                    maldi_pixel_um = 10.0
                    _log(f"  Falling back to maldi_pixel_um=10.0 um/px.")
        else:
            maldi_pixel_um = 10.0
            _log(f"  Pixel size not in imzML, defaulting to {maldi_pixel_um} um/px.")
    else:
        _log(f"  MALDI pixel size (supplied): {maldi_pixel_um} um/px")

    _log(f"  maldi_pixel_um={maldi_pixel_um}  he_pixel_um={he_pixel_um:.4f}")

    # ------------------------------------------------------------------
    # 2. MALDI crop offsets
    # ------------------------------------------------------------------
    _log("Computing MALDI crop offsets ...")
    tic_probe = np.nansum(
        np.stack([getimage(pk, path=imzml_path) for pk in peaks[:5]], axis=-1),
        axis=-1,
    )
    crop_r, crop_c = _crop_offsets(tic_probe)
    _log(f"  Crop: row={crop_r}, col={crop_c}")
    del tic_probe
    gc.collect()

    # ------------------------------------------------------------------
    # 3. Load spectra in chunks
    # ------------------------------------------------------------------
    _log(f"Loading {len(peaks)} ion images (chunk={spectra_chunk_size}) ...")
    spectra_all = _load_spectra(
        imzml_path, peaks,
        chunk_size=spectra_chunk_size,
        crop_r=crop_r, crop_c=crop_c,
    )
    _log(f"  spectra_all: {spectra_all.shape}  ({spectra_all.nbytes/1e6:.0f} MB)")

    # ------------------------------------------------------------------
    # 4. MALDI registration image
    # ------------------------------------------------------------------
    _log("Preparing MALDI template ...")
    maldi_tic  = spectra_all.sum(axis=-1).astype(np.float32)
    maldi_gray = _maldi_to_grayscale(maldi_tic)
    del maldi_tic
    gc.collect()

    # ------------------------------------------------------------------
    # 5. H&E at MALDI native resolution
    # ------------------------------------------------------------------
    _log(f"Loading H&E at {maldi_pixel_um} um/px ...")
    he_img, loaded_mpp = _load_he_at_resolution(he_path, maldi_pixel_um, he_pixel_um)
    _log(f"  H&E: {he_img.width}x{he_img.height}  ({he_img.width*he_img.height*3/1e6:.0f} MB)")

    # he_reg_w / he_reg_h are the H&E dimensions AT reg_mpp (already downscaled
    # from native). These are passed directly to _pil_rotation_affine as the
    # source image dimensions — they define the rotation centre and the
    # bounding-box expansion that PIL actually performs.
    he_reg_w = he_img.width
    he_reg_h = he_img.height

    # ------------------------------------------------------------------
    # 6. H&E registration image
    # ------------------------------------------------------------------
    _log("Preparing H&E search image ...")
    he_gray = _he_to_grayscale(he_img)

    # ------------------------------------------------------------------
    # 7. Two-pass rotation + translation search
    # ------------------------------------------------------------------
    _log("Running registration ...")
    best_rot, best_idx = _register(
        he_gray, maldi_gray,
        coarse_step=coarse_rotation_step,
        fine_range=fine_rotation_range,
        fine_step=fine_rotation_step,
        buffer_px=buffer_px,
    )
    del he_gray, maldi_gray
    gc.collect()

    # ------------------------------------------------------------------
    # 8. Build full H&E output canvas.
    # ------------------------------------------------------------------
    _log("Building H&E output canvas ...")
    he_canvas, canvas_pr, canvas_pc = _build_full_output(
        he_img    = he_img,
        rotation  = best_rot,
        buffer_px = buffer_px,
    )
    del he_img
    gc.collect()

    return(he_gray, maldi_gray,he_canvas, canvas_pr,canvas_pc, geojson_path,
            he_pixel_um,
            loaded_mpp,
            he_reg_w  ,
            he_reg_h        ,
            best_rot      ,
            canvas_pr         ,
            canvas_pc         ,
            img_upscaling     ,
            geojson_classification_key)
    # ------------------------------------------------------------------
    # 9. Transform annotations.
    #    he_reg_w / he_reg_h are passed directly — NOT rescaled again.
    # ------------------------------------------------------------------
    annotation_gdf    = None
    annotation_matrix = None
    if geojson_path is not None:
        _log(f"Transforming annotations from {geojson_path} ...")
        annotation_gdf, annotation_matrix = _transform_geojson(
            geojson_path       = geojson_path,
            he_pixel_um        = he_pixel_um,
            reg_mpp            = loaded_mpp,
            he_reg_w           = he_reg_w,
            he_reg_h           = he_reg_h,
            rotation_deg       = best_rot,
            canvas_pr          = canvas_pr,
            canvas_pc          = canvas_pc,
            img_upscaling      = img_upscaling,
            classification_key = geojson_classification_key,
        )
        unique = annotation_gdf[geojson_classification_key].unique().tolist()
        _log(f"  {len(annotation_gdf)} annotations  |  classes: {unique}")

    # ------------------------------------------------------------------
    # 10. Assemble SpatialData
    # ------------------------------------------------------------------
    _log("Building SpatialData ...")
    sdata = _build_spatialdata(
        spectra_all            = spectra_all,
        peaks                  = peaks,
        maldi_pixel_um         = maldi_pixel_um,
        he_canvas              = he_canvas,
        maldi_offset_in_canvas = best_idx,
        reg_mpp                = loaded_mpp,
        crop_r                 = crop_r,
        crop_c                 = crop_c,
        img_upscaling          = img_upscaling,
    )

    if annotation_gdf is not None:
        ann_shapes = ShapesModel.parse(
            annotation_gdf,
            transformations={"global": Identity()},
        )
        sdata.shapes[geojson_shapes_key] = ann_shapes
        _log(f"  Annotations added -> sdata.shapes['{geojson_shapes_key}']")

    # ------------------------------------------------------------------
    # 11. Store registration transform + affine matrix.
    #     affine_matrix maps native H&E px -> final upscaled canvas and is
    #     used by add_qupath_annotations() for any subsequent GeoJSON files.
    # ------------------------------------------------------------------
    if annotation_matrix is None:
        annotation_matrix = _build_full_transform_matrix(
            he_pixel_um  = he_pixel_um,
            reg_mpp      = loaded_mpp,
            he_reg_w     = he_reg_w,
            he_reg_h     = he_reg_h,
            rotation_deg = best_rot,
            canvas_pr    = canvas_pr,
            canvas_pc    = canvas_pc,
            img_upscaling= img_upscaling,
        )

    sdata["maldi_adata"].uns["he_transform"] = {
        "rotation_deg":     float(best_rot),
        "maldi_offset":     [int(best_idx[0]), int(best_idx[1])],
        "he_pixel_um":      float(he_pixel_um),
        "maldi_pixel_um":   float(maldi_pixel_um),
        "reg_mpp":          float(loaded_mpp),
        "buffer_px":        int(buffer_px),
        "img_upscaling":    int(img_upscaling),
        "canvas_shape":     list(he_canvas.shape[:2]),
        "he_reg_size":      [int(he_reg_h), int(he_reg_w)],
        "canvas_placement": [int(canvas_pr), int(canvas_pc)],
        "affine_matrix":    annotation_matrix.tolist(),
    }

    _log(f"  Transform stored: he_reg_size={[he_reg_h, he_reg_w]}  "
         f"canvas_placement={[canvas_pr, canvas_pc]}")
    _log("Done.")
    return sdata

In [5]:
he_gray, maldi_gray,he_canvas, canvas_pr,canvas_pc, geojson_path,he_pixel_um,loaded_mpp,he_reg_w  ,he_reg_h        ,best_rot      ,canvas_pr         ,            canvas_pc         ,img_upscaling     ,geojson_classification_key = load_and_align(imzml_path = "/Users/andrewcauser/Documents/Griffith/test_data/breast_cancer_concr_0019/concr_tnbc_0019-s1-total_ion_count.imzML",
                peaks_path = "/Users/andrewcauser/Documents/Griffith/breast_peaks_formatted.csv",
                he_path="/Users/andrewcauser/Documents/Griffith/test_data/breast_cancer_concr_0019/tnbc_0019.svs",
               geojson_path = "/Users/andrewcauser/Documents/Griffith/test_data/breast_cancer_concr_0019/tnbc_0019_annotations.geojson")

[0.30GB] Loading peaks ...
[0.30GB]   121 peaks
[0.31GB]   H&E native pixel size: 0.2527 um/px
[0.28GB]   MALDI pixel size from imzML metadata: 50.0 um/px
[0.60GB]   Validated: H&E thumbnail (553x389 px) >= MALDI (542x378 px)
[0.60GB]   maldi_pixel_um=50.0  he_pixel_um=0.2527
[0.60GB] Computing MALDI crop offsets ...
[0.59GB]   Crop: row=0, col=0
[0.58GB] Loading 121 ion images (chunk=10) ...


/Users/andrewcauser/anaconda3/envs/maldi_env/lib/python3.10/site-packages/pyimzml/ontology/ontology.py:92: UserWarning: Accession IMS:1000046 found with incorrect name "pixel size x". Updating name to "pixel size (x)".
  warn(
/Users/andrewcauser/anaconda3/envs/maldi_env/lib/python3.10/site-packages/pyimzml/ontology/ontology.py:92: UserWarning: Accession IMS:1000046 found with incorrect name "pixel size x". Updating name to "pixel size (x)".
  warn(
/Users/andrewcauser/anaconda3/envs/maldi_env/lib/python3.10/site-packages/pyimzml/ontology/ontology.py:92: UserWarning: Accession IMS:1000046 found with incorrect name "pixel size x". Updating name to "pixel size (x)".
  warn(
/Users/andrewcauser/anaconda3/envs/maldi_env/lib/python3.10/site-packages/pyimzml/ontology/ontology.py:92: UserWarning: Accession IMS:1000046 found with incorrect name "pixel size x". Updating name to "pixel size (x)".
  warn(


[0.15GB]   Peaks 1-10 / 121
[0.15GB]   Peaks 11-20 / 121


/Users/andrewcauser/anaconda3/envs/maldi_env/lib/python3.10/site-packages/pyimzml/ontology/ontology.py:92: UserWarning: Accession IMS:1000046 found with incorrect name "pixel size x". Updating name to "pixel size (x)".
  warn(


[0.15GB]   Peaks 21-30 / 121


/Users/andrewcauser/anaconda3/envs/maldi_env/lib/python3.10/site-packages/pyimzml/ontology/ontology.py:92: UserWarning: Accession IMS:1000046 found with incorrect name "pixel size x". Updating name to "pixel size (x)".
  warn(


KeyboardInterrupt: 

In [ ]:
interactive = Interactive(sdata)
viewer = interactive.run()